# mIF Pipeline: Full-Slide InstanSeg/Nimbus API Debug Notebook

This notebook mirrors the crop notebook but points at the full-slide prototype YAML. Use it in the `instanseg_nimbus` environment for interactive prep across selected slides and then run one target slide through merge, InstanSeg, and Nimbus.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "prototyping").exists():
            return candidate
    raise RuntimeError("Could not find the repository root containing src/ and prototyping/.")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))


def show(data):
    if isinstance(data, str):
        print(data)
    else:
        print(json.dumps(data, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"=== {label} ===")
    result = func(*args, **kwargs)
    show(result)
    return result

from mif_pipeline import (
    merge_slide_ometiffs,
    prepare_nimbus_normalization,
    run_instanseg,
    run_nimbus_chunked,
    setup_slides,
)
from mif_pipeline.config import get_slide_config, load_config


In [ ]:
CONFIG_PATH = REPO_ROOT / "prototyping" / "prototype_v2-fullslide.yaml"
PREP_SLIDE_IDS = ["SLIDE-0329"]
TARGET_SLIDE_ID = PREP_SLIDE_IDS[0]

SETUP_FORCE = False
NIMBUS_PREP_FORCE = False
MERGE_FORCE = False
INSTANSEG_FORCE = False
NIMBUS_FORCE = False

NIMBUS_PREP_CHUNKS = None
NIMBUS_RUN_CHUNKS = None


In [ ]:
config = load_config(CONFIG_PATH)
print("config:", CONFIG_PATH)
print("prep slides:", PREP_SLIDE_IDS)
print("target slide:", TARGET_SLIDE_ID)


In [ ]:
resolved_slides = {slide_id: get_slide_config(config, slide_id) for slide_id in PREP_SLIDE_IDS}
for slide_id, slide in resolved_slides.items():
    print(f"
[{slide_id}]")
    print("slide_dir:", slide["slide_dir"])
    print("output_dir:", slide["output_dir"])
    print("full_merge_path:", slide["full_merge"]["ome_path"])
    print("nimbus_output_dir:", slide["nimbus"]["output_dir"])


In [ ]:
setup_results = stage_call(
    f"setup_slides({PREP_SLIDE_IDS}, force={SETUP_FORCE})",
    setup_slides,
    config,
    slide_ids=PREP_SLIDE_IDS,
    force=SETUP_FORCE,
)


In [ ]:
nimbus_prep_results = stage_call(
    f"prepare_nimbus_normalization({PREP_SLIDE_IDS}, dry_run=False)",
    prepare_nimbus_normalization,
    config,
    PREP_SLIDE_IDS,
    chunk_indices=NIMBUS_PREP_CHUNKS,
    force=NIMBUS_PREP_FORCE,
)


In [ ]:
merge_results = stage_call(
    f"merge_slide_ometiffs({TARGET_SLIDE_ID}, dry_run=False)",
    merge_slide_ometiffs,
    config,
    TARGET_SLIDE_ID,
    force=MERGE_FORCE,
)


In [ ]:
instanseg_results = stage_call(
    f"run_instanseg({TARGET_SLIDE_ID}, dry_run=False)",
    run_instanseg,
    config,
    TARGET_SLIDE_ID,
    force=INSTANSEG_FORCE,
)


In [ ]:
nimbus_results = stage_call(
    f"run_nimbus_chunked({TARGET_SLIDE_ID}, dry_run=False)",
    run_nimbus_chunked,
    config,
    TARGET_SLIDE_ID,
    chunk_indices=NIMBUS_RUN_CHUNKS,
    force=NIMBUS_FORCE,
)


In [ ]:
target_slide = get_slide_config(config, TARGET_SLIDE_ID)
target_nimbus_dir = Path(target_slide["nimbus"]["output_dir"])
print("merged cell table:", target_nimbus_dir / "cell_table_full.csv")
for chunk_dir in sorted(target_nimbus_dir.glob("chunk_*")):
    print(chunk_dir)
